### Benchmark downloader
The script filters and downloads benchmarks from MIPLIB 2017. Please see guide and example of usage at the bottom of the script

In [ ]:
import pandas as pd
import requests
import pathlib
import time

In [ ]:
DOWNLOAD_URL_TEMPLATE = "https://miplib.zib.de/WebData/instances/%s.mps.gz"
DOWNLOAD_FILE_TEMPLATE = "%s.mps.gz"
CSV_INFO_FILENAME = "The Benchmark Set.csv"
DEFAULT_OUTPUT_FOLDER = "../benchmarks/"
CHUNK_SIZE = 1024

In [ ]:
def filter_benchmarks(
    csv_info_filename: str,
    min_vars: int,
    max_vars: int,
    min_int_vars: int,
    max_int_vars: int,
    min_cont_vars: int,
    max_cont_vars: int,
    min_bin_vars: int,
    max_bin_vars: int,
    all_tags: bool,
    tags: str
) -> pd.DataFrame:
    df = pd.read_csv(csv_info_filename)
    filtrated = df[df["Objective  Obj."].str.contains(r'\d[.]\d') == False]

    if max_vars >= 0:
        filtrated = filtrated[filtrated["Variables  Var."] <= max_vars]
    if min_vars >= 0:
        filtrated = filtrated[filtrated["Variables  Var."] >= min_vars]
    if max_int_vars >= 0:
        filtrated = filtrated[filtrated["Integers  Int."] <= max_int_vars]
    if min_int_vars >= 0:
        filtrated = filtrated[filtrated["Integers  Int."] >= min_int_vars]
    if max_cont_vars >= 0:
        filtrated = filtrated[filtrated["Continuous  Con."] <= max_cont_vars]
    if min_cont_vars >= 0:
        filtrated = filtrated[filtrated["Continuous  Con."] >= min_cont_vars]
    if max_bin_vars >= 0:
        filtrated = filtrated[filtrated["Binaries  Bin."] <= max_bin_vars]
    if min_bin_vars >= 0:
        filtrated = filtrated[filtrated["Binaries  Bin."] >= min_bin_vars]

    if tags != "":
        if all_tags:
            # Filter benchmarks containing all specified tags
            tag_list = tags.split(",")
            base = r'^{}'
            expr = '(?=.*{})'
            tags_regex = base.format(''.join(expr.format(w) for w in tag_list))
        else:
            # Filter benchmarks containing any specified tag
            tags_regex = tags.replace(",", "|")
        filtrated = filtrated.loc[filtrated["Tags  Tag."].str.contains(tags_regex) == True]
    return filtrated

In [ ]:
def download_benchmarks(
    csv_info_filename: str = CSV_INFO_FILENAME,
    output_folder: str = DEFAULT_OUTPUT_FOLDER,
    min_vars: int = -1,
    max_vars: int = -1,
    min_int_vars: int = -1,
    max_int_vars: int = -1,
    min_cont_vars: int = -1,
    max_cont_vars: int = -1,
    min_bin_vars: int = -1,
    max_bin_vars: int = -1,
    all_tags: bool = True,
    tags: str = ""
):
    """
    Download benchmarks filtered by specified constraints.

    Args:
        csv_info_filename: filename of cvs file with benchmarks info
        output_folder: folder name to store downloads
        min_vars: min number of all variables
        max_vars: max number of all variables
        min_int_vars: minimum number of integer variables
        max_int_vars: max number of integer variables
        min_cont_vars: minimum number of continuous variables
        max_cont_vars: max number of continuous variables
        min_bin_vars: minimum number of binary variables
        max_bin_vars: max number of binary variables
        all_tags: whether to use AND or OR operator between tags
        tags: list of tags divided by comma
    """
    output_folder = output_folder if output_folder[-1] == '/' else output_folder + '/'
    pathlib.Path(output_folder).mkdir(parents=True, exist_ok=True)

    benchmarks_df = filter_benchmarks(
        csv_info_filename,
        min_vars,
        max_vars,
        min_int_vars,
        max_int_vars,
        min_cont_vars,
        max_cont_vars,
        min_bin_vars,
        max_bin_vars,
        all_tags,
        tags
    )

    size = len(benchmarks_df.index)
    print(f"Found {size} benchmarks:")
    print(benchmarks_df["Instance  Ins."].to_numpy())
    for i, (_, row) in enumerate(benchmarks_df.iterrows()):
        url = DOWNLOAD_URL_TEMPLATE % row["Instance  Ins."]
        save_path = output_folder + DOWNLOAD_FILE_TEMPLATE % f'{row["Instance  Ins."]}_vc{row["Variables  Var."]}'
        response = requests.get(url, stream=True)

        with open(save_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                file.write(chunk)

        # Wait for a couple of seconds to avoid being blocked by the server
        if i % 10 == 0:
            print(f"progress: {i+1}/{size}")
            time.sleep(2)
        else:
            time.sleep(1)
    print(f"progress: {size}/{size}")
    print(f"All benchmarks are downloaded into folder {DEFAULT_OUTPUT_FOLDER}")

### Guide:
The script downloads benchmarks from MIPLIB 2017. The file with benchmark info "The Benchmark Set.csv" downloaded manually from MIPLIB 2017. 
To filter and download benchmarks call function `download_benchmarks` and pass a list of parameters (used for filtering):
1. Specify range of all, continuous and binary variables using the following arguments: 
- min_vars
- max_vars
- min_int_vars
- max_int_vars
- min_cont_vars
- max_cont_vars
- min_bin_vars
- max_bin_vars
2. Specify tag parameters:
- all_tags - whether to use AND or OR operator between tags
- tags - list of tags divided by comma

If parameter is set to negative value or empty string, then this filter will not be applied

In [ ]:
min_vars = 0
max_vars = 200
min_int_vars = 0
max_int_vars = 10000
min_cont_vars = 0
max_cont_vars = 10000
min_bin_vars = 0
max_bin_vars = 10000
all_tags = False  # True - AND operator between tags, False - OR operator
tags = "equation_knapsack,general_linear"

download_benchmarks(
    CSV_INFO_FILENAME,
    DEFAULT_OUTPUT_FOLDER,
    min_vars,
    max_vars,
    min_int_vars,
    max_int_vars,
    min_cont_vars,
    max_cont_vars,
    min_bin_vars,
    max_bin_vars,
    all_tags,
    tags
)